# 00 · Recap: where the project stands

A guided tour of everything built so far, meant for catching up after a break. Each part
explains one layer of the code, then runs it on real data, so the outputs double as a
reminder of what the functions return.

| Part | Layer | Code |
|---|---|---|
| 1 | The pipeline at a glance | |
| 2 | Playing battles in SMAClite | `sindy_wm.envs` |
| 3 | Collecting and storing datasets | `sindy_wm.io` |
| 4 | The experiments on disk (exp01–exp03) | `sindy_wm.evaluation.sweep`, `scripts/` |
| 5 | From snapshots to states | `sindy_wm.states.aggregate` |
| 6 | Modelling: SINDy and Lanchester | `sindy_wm.models` |
| 7 | Saved runs | `runs/` |
| 8 | Where everything lives (notebooks, function index) | |
| 9 | Findings, open questions, loose ends | |

Everything runs from the project's `.venv` kernel, in about a minute. Nothing in `data/raw/` or
`runs/` is written; Part 3 collects a tiny demo dataset in a temporary folder.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import inspect
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps
from matplotlib.patches import Rectangle

import sindy_wm
from sindy_wm.envs.episode import play_episode
from sindy_wm.envs.policies import AttackNearestPolicy
from sindy_wm.envs.scenarios import (
    builtin_scenarios,
    custom_scenarios,
    load_map_info,
    make_env,
    walkable_bounds,
)
from sindy_wm.envs.spawn_sampling import sample_opposing_spawns
from sindy_wm.evaluation.plots import TEAM_COLORS, plot_level_a, plot_positions, plot_predictions
from sindy_wm.evaluation.sweep import heatmap, load_all_indexes, summarize
from sindy_wm.io import storage
from sindy_wm.io.collect import collect
from sindy_wm.models.lanchester import fit_lanchester
from sindy_wm.models.metrics import summarize_eval
from sindy_wm.models.simulation import simulate_to_elimination
from sindy_wm.models.sindy import fit_sindy
from sindy_wm.models.trajectories import to_trajectory
from sindy_wm.paths import PROJECT_ROOT, RAW_DIR
from sindy_wm.states.aggregate import engagement, level_a

plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 160)

# The dataset used in Parts 5 and 6 (the same one notebook 04 currently fits)
EXPERIMENT = "exp03_10m_random_aggression_stepmul"
DATASET = "10m_attack_a090_s1_v1"
RUNS_DIR = PROJECT_ROOT / "runs"

## Part 1 · The pipeline at a glance

The thesis question: can SINDy learn a **compact, interpretable model of combat dynamics** from
simulated battles, good enough to later serve as a world model for planning?

The code is a straight pipeline. Each arrow is one module:

```
SMAClite battle ──envs──▶ Episode (per-unit snapshots + actions)
                ──io────▶ dataset folder in data/raw/<experiment>/<dataset>/
                ──states▶ Level A table: health/alive/distance per team per step
                ──models▶ to_trajectory → fit_sindy / fit_lanchester → evaluate
                ──save──▶ runs/<date>_<name>/ (config, equations, metrics, figures)
```

Three principles hold it together (from the README's Conventions):

- **Log raw snapshots once, derive everything else.** States are never stored, so a new state
  definition never needs new games.
- **Everything is reproducible from a seed.** The same seed drives the environment, the policy and
  the spawn sampler; every dataset and run records the git commit that made it.
- **Raw data and runs are never overwritten.** New results get new names.

The same pipeline in a few lines, for one episode:

```python
env = make_env("5m_vs_5m", step_mul=8)
episode = play_episode(env, AttackNearestPolicy(aggression=0.9), seed=0)
states = level_a(episode.snapshots, normalize=True)
```

The rest of the notebook goes through each step in detail.

## Part 2 · Playing battles in SMAClite (`sindy_wm.envs`)

[SMAClite](https://github.com/uoe-agents/smaclite) is a light Python version of the StarCraft
Multi-Agent Challenge. **Blue** (allies) is controlled by our policy, **Red** (enemies) by the
game's built-in AI. Only `envs/` imports `smaclite`; everything downstream works on tables.

### 2.1 Scenarios and `step_mul`

A scenario is either a SMAClite built-in map or one of our JSON files in `configs/scenarios/`.
We use our own `5m_vs_5m` and `10m_vs_10m` (marines against marines, open terrain).

`step_mul` is the number of game ticks per `env.step` (16 ticks = 1 s). It sets both how finely
a battle is recorded and how often Blue gets to decide. `make_env` wraps the environment so each
one keeps its own `step_mul` (SMAClite stores it as a global).

In [ ]:
print("Our scenarios:     ", custom_scenarios())
print("SMAClite built-in: ", builtin_scenarios())

env = make_env("5m_vs_5m", step_mul=8)
print(f"\nstep_mul {env.step_mul} -> {env.seconds_per_step} s per step")

### 2.2 Policies and one episode

A policy returns one action per Blue unit each step. SMAClite's actions: 0 no-op, 1 stop,
2–5 move north/south/east/west, 6+ attack enemy *k*.

- `RandomPolicy`: uniformly random among available actions.
- `AttackNearestPolicy(aggression)`: attack the nearest enemy in range, otherwise walk towards
  the nearest one. With probability `1 - aggression` a unit takes a random action instead, which
  is the main source of variety between episodes. (Units never fire on their own; a policy must
  choose to attack.)

`play_episode` plays until one side is dead or `max_steps` is reached (SMAClite has no time limit
of its own), and returns an `Episode` with two tables:

- `snapshots`: one row per unit per step. Dead units stay in the table with `hp = 0`.
- `actions`: Blue's action per unit per step.

In [ ]:
episode = play_episode(env, AttackNearestPolicy(aggression=0.9), seed=0)
print(f"{episode.n_steps} steps, battle_won={episode.battle_won}, truncated={episode.truncated}")
episode.snapshots.head(10)

In [ ]:
episode.actions.head(10)

`plot_level_a` shows the team-level summary (Part 5 explains `level_a`), and `plot_positions`
the map at a single step. For an animation, use `animate_positions(episode.snapshots)` with
`plt.rcParams["animation.html"] = "jshtml"`.

In [ ]:
plot_level_a(level_a(episode.snapshots), title="5m_vs_5m, aggression 0.9, seed 0")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, step in zip(axes, [0, episode.n_steps // 2], strict=True):
    at = episode.snapshots[(episode.snapshots["step"] == step) & episode.snapshots["alive"]]
    for team, color in TEAM_COLORS.items():
        units = at[at["team"] == team]
        ax.scatter(units["x"], units["y"], color=color, label=team)
    ax.set(title=f"step {step}", aspect="equal", xlabel="x", ylabel="y")
    ax.grid(True, alpha=0.3)
axes[0].legend()
fig.tight_layout()
plt.show()

### 2.3 Randomized starting positions

By default every episode of a scenario starts from the same positions. With a randomized spawn,
`sample_opposing_spawns` draws a new **distance and bearing** between the two groups per episode
(each team keeps its formation; both are centred on the middle of the map). It is used by
`collect(..., randomize_spawn=True)`, and was introduced for exp02 and exp03.

Each line below is one sampled episode start; the box is the walkable area.

In [ ]:
bounds = walkable_bounds(load_map_info("5m_vs_5m"))
rng = np.random.default_rng(0)
xmin, xmax, ymin, ymax = bounds

fig, ax = plt.subplots(figsize=(8, 4))
ax.add_patch(Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color="0.5", ls="--"))
for _ in range(30):
    s = sample_opposing_spawns(rng, bounds, min_distance=4.0, max_distance=16.0)
    ax.plot(*zip(s.ally_pos, s.enemy_pos, strict=True), color="0.75", lw=0.8)
    ax.scatter(*s.ally_pos, color=TEAM_COLORS["blue"], s=15)
    ax.scatter(*s.enemy_pos, color=TEAM_COLORS["red"], s=15)
ax.set(aspect="equal", title="30 sampled spawns (distance 4-16)", xlabel="x", ylabel="y")
plt.show()

spawn = sample_opposing_spawns(np.random.default_rng(1), bounds, 4.0, 16.0)
random_start = play_episode(make_env("5m_vs_5m", step_mul=8, spawn=spawn), AttackNearestPolicy(0.9), seed=1)
plot_positions(random_start.snapshots, step=0, title="one randomized start, step 0")
plt.show()

### 2.4 Reproducibility

Same seed, same scenario, same policy → the same episode, unit for unit. (Copying a running game
with `deepcopy` does *not* reproduce it; replaying from seed and action log does. See the README.)

In [ ]:
runs = [
    play_episode(make_env("5m_vs_5m", step_mul=8), AttackNearestPolicy(aggression=0.7), seed=3)
    for _ in range(2)
]
pd.testing.assert_frame_equal(runs[0].snapshots, runs[1].snapshots)
print("Two plays of seed 3 are identical:", runs[0].n_steps, "steps each")

## Part 3 · Collecting and storing datasets (`sindy_wm.io`)

A **dataset** is many episodes with one configuration (scenario, `step_mul`, policy). `collect()`
plays them with consecutive seeds and saves them through `storage`:

```
<dataset>/
    manifest.json      how it was made: config, seeds, git commit, package versions, status
    episodes.jsonl     one line per episode: seed, n_steps, battle_won, final alive/hp per team
    snapshots/episode_00000.parquet ...
    actions/episode_00000.parquet ...
```

`create_dataset` refuses an existing folder and, unless `allow_dirty=True`, a git tree with
uncommitted changes (so the recorded commit really describes the code). `finalize_dataset`
checks that every seed was saved and marks the dataset `complete`.

Below, a 3-episode demo dataset in a temporary folder (not in `data/raw/`):

In [ ]:
demo_dir = Path(tempfile.mkdtemp()) / "demo_5m_attack_a090_s8"
collect(
    demo_dir,
    policy=AttackNearestPolicy(aggression=0.9),
    description="recap notebook demo",
    scenario="5m_vs_5m",
    step_mul=8,
    n_episodes=3,
    allow_dirty=True,
    randomize_spawn=True,
)
print()
for p in sorted(demo_dir.rglob("*")):
    print(p.relative_to(demo_dir))

In [ ]:
manifest = storage.load_manifest(demo_dir)
print(json.dumps({k: manifest[k] for k in ["name", "status", "n_episodes", "config"]}, indent=2))
print("git commit:", manifest["code"]["git_commit"], " dirty:", manifest["code"]["git_dirty"])
storage.load_episode_index(demo_dir)

Reading back is the same for any dataset: `load_manifest`, `load_episode_index`,
`load_snapshots(dataset_dir)` (all episodes, with an `episode_idx` column) and `load_actions`.

The scripts in `scripts/` call `collect()` over a grid of settings, one script per experiment:
`step_aggression_sweep.py` made exp01, `collect_data.py` is the one edited for each new
experiment (currently exp03), and `how_to_store.py` is a walk-through of the storage API without
`collect()`. They skip datasets that already exist, so they can be rerun after a crash.

## Part 4 · The experiments on disk

Each experiment is a folder in `data/raw/` with one dataset per setting, all with
`AttackNearestPolicy` against Red's built-in AI, over the grid
aggression ∈ {0.5, 0.7, 0.9, 1.0} × `step_mul` ∈ {1, 2, 4, 8}.

In [ ]:
rows = []
for exp in sorted(p for p in RAW_DIR.iterdir() if p.is_dir()):
    datasets = sorted(d for d in exp.iterdir() if (d / "manifest.json").exists())
    if not datasets:
        rows.append({"experiment": exp.name, "datasets": 0})
        continue
    m = storage.load_manifest(datasets[0])
    first_snaps = storage.load_snapshots(datasets[0], 0)
    rows.append(
        {
            "experiment": exp.name,
            "datasets": len(datasets),
            "scenario": m["config"]["scenario"]["name"],
            "randomized_spawn": "spawn_randomization" in m["config"]["scenario"],
            "episodes_per_dataset": m["n_episodes"],
            "has_target_velocity_columns": "target_in_range" in first_snaps.columns,
        }
    )
experiments = pd.DataFrame(rows)
experiments

The table is read from each dataset's own `manifest.json`, so it shows what is really on disk:

- **exp01** (`5m_vs_5m`, fixed spawn): the first sweep. At aggression 1.0 every episode is the same
  game, because nothing is random.
- **exp02** (`5m_vs_5m`, randomized spawn): the same grid with random distance and bearing, which
  makes every setting varied (compared with exp01 in notebook 05).
- **exp03** (`10m_vs_10m`, randomized spawn): twice the units, used in notebooks 04 and 06.

Both 5m experiments were collected again after their first findings were written; the README's
Findings are recomputed on the data now on disk (see its "What is on disk" table).

Blue win rate per setting, for each experiment:

In [ ]:
summaries = {}
for exp in experiments.loc[experiments["datasets"] > 0, "experiment"]:
    episodes = load_all_indexes(RAW_DIR / exp)
    summaries[exp] = (
        episodes.groupby(["dataset", "step_mul", "aggression"])
        .apply(summarize, include_groups=False)
        .reset_index()
    )

fig, axes = plt.subplots(1, len(summaries), figsize=(5.5 * len(summaries), 4))
for ax, (exp, summary) in zip(np.atleast_1d(axes), summaries.items(), strict=True):
    _, _, table = heatmap(summary, "win_rate", exp.split("_")[0] + " " + exp.split("_")[1], ax=ax)
    for i, a in enumerate(table.index):
        for j, s in enumerate(table.columns):
            v = table.loc[a, s]
            ax.text(
                j,
                i,
                f"{v:.0%}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if v < 0.2 or v > 0.8 else "black",
            )
fig.tight_layout()
plt.show()

Battle length in seconds is about the same at every `step_mul`: `step_mul` changes how often
Blue decides and how finely the battle is recorded, not the battle itself.

In [ ]:
pd.concat(
    {
        exp: s.pivot_table(index="aggression", columns="step_mul", values="duration_mean_s").round(1)
        for exp, s in summaries.items()
    },
    names=["experiment"],
)

## Part 5 · From snapshots to states (`sindy_wm.states.aggregate`)

SINDy needs a small state vector, not 20 units × 15 columns. Two functions compress a snapshot
table to one row per step:

- `level_a(snaps, normalize=True)`: **Level A**, the fully aggregated state.
  `health_*` (hp + shield summed over a team), `alive_*` (number of alive units), and
  `distance` between the two teams' centroids. With `normalize=True`, health and alive start at 1.
- `engagement(snaps)`: per team, `engaged_*` (alive units with a target in range, i.e. actually
  shooting), `targets_*` (distinct enemies being shot at; 1 = focused fire) and `speed_*`. Only
  for exp03-style snapshots with `target_*`/`vx`/`vy` columns.

Loading `DATASET` (see the setup cell) and computing both for every episode:

In [ ]:
dataset_dir = RAW_DIR / EXPERIMENT / DATASET
manifest = storage.load_manifest(dataset_dir)
index = storage.load_episode_index(dataset_dir).set_index("episode_idx")
dt = manifest["config"]["env"]["seconds_per_step"]
snaps = storage.load_snapshots(dataset_dir)

states = snaps.groupby("episode_idx").apply(level_a, normalize=True, include_groups=False)
states = states.join(snaps.groupby("episode_idx").apply(engagement, include_groups=False))
states["t"] = states.index.get_level_values("step") * dt

print(f"{DATASET}: {len(index)} episodes, dt = {dt} s, Blue win rate {index['battle_won'].mean():.0%}")
states.loc[0].iloc[::40].head(8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)
for ax, won, label in [(axes[0], True, "Blue won"), (axes[1], False, "Blue lost")]:
    for i in index.index[index["battle_won"] == won][:40]:
        ep = states.loc[i]
        for team, color in TEAM_COLORS.items():
            ax.plot(ep["t"], ep[f"health_{team}"], color=color, alpha=0.25, lw=1)
    ax.set(title=f"{label} (first 40 episodes)", xlabel="time (s)")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("health (normalized)")
fig.tight_layout()
plt.show()

ep = states.loc[0]
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for team, color in TEAM_COLORS.items():
    axes[0].plot(ep["t"], ep[f"health_{team}"], color=color, label=team)
    axes[1].plot(ep["t"], ep[f"engaged_{team}"], color=color)
    axes[2].plot(ep["t"], ep[f"targets_{team}"], color=color)
axes[0].set_ylabel("health")
axes[1].set_ylabel("engaged units")
axes[2].set(ylabel="targets", xlabel="time (s)")
axes[0].set_title("episode 0: health against engagement")
axes[0].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

Notice the flat start: while the teams walk towards each other, nothing happens to health.
That approach phase is why trajectories are trimmed before fitting (next part).

## Part 6 · Modelling: SINDy and Lanchester (`sindy_wm.models`)

### 6.1 The idea

**SINDy** (Sparse Identification of Nonlinear Dynamics) assumes $\dot x = \Theta(x)\,\Xi$:
the derivative of the state is a *sparse* combination of candidate terms $\Theta(x)$ (here a
polynomial library: $1, B, R, B^2, BR, R^2, \dots$). It estimates $\dot x$ from data, then runs a
sparse regression (STLSQ: least squares, zero out coefficients below `threshold`, repeat).

**Lanchester's square law** (1916) is the classical model it can be compared with:
$\dot B = -a R$, $\dot R = -b B$: each side's losses are proportional to the other side's
strength. `fit_lanchester` fits exactly this 2-parameter form, as a baseline.

### 6.2 Preparing trajectories

`to_trajectory` turns one episode's Level A table into the rows used for fitting:

- **trim to engagement**: start one step before the first damage (drops the approach phase),
- **downsample**: keep every *k*-th row, so one point covers several shots,
- reset `t` to 0.

Then a **train/test split by episode** (never by row, and episodes are never stacked into one
array, because the jump between them would look like a huge derivative).

In [ ]:
STATE_COLUMNS = ["health_blue", "health_red"]  # Blue first, Red second: evaluate() relies on it
DOWNSAMPLE = 2  # at step_mul 1: 1/8 s per point
SMOOTH_WINDOW = 9  # Savitzky-Golay window for the derivative, in (downsampled) points

trajectories = {i: to_trajectory(states.loc[i], STATE_COLUMNS, True, DOWNSAMPLE) for i in index.index}
dt_fit = dt * DOWNSAMPLE

rng = np.random.default_rng(0)  # same split as notebook 04
episode_ids = rng.permutation(index.index.to_numpy())
n_test = int(round(0.2 * len(episode_ids)))
test_ids = sorted(episode_ids[:n_test].tolist())
train_ids = sorted(episode_ids[n_test:].tolist())
print(f"{len(train_ids)} training and {len(test_ids)} test episodes, dt = {dt_fit} s")

ep_id = test_ids[0]
raw_ep = states.loc[ep_id]
trimmed_rows = len(to_trajectory(raw_ep, STATE_COLUMNS, True, downsample=1))
t0 = raw_ep["t"].iloc[len(raw_ep) - trimmed_rows]  # where the trimmed trajectory starts
traj = trajectories[ep_id]
fig, ax = plt.subplots(figsize=(8, 3))
for team, color in TEAM_COLORS.items():
    ax.plot(raw_ep["t"], raw_ep[f"health_{team}"], color=color, alpha=0.3, label=f"{team}, raw")
    ax.plot(traj["t"] + t0, traj[f"health_{team}"], ".", color=color, ms=3, label=f"{team}, trajectory")
ax.axvline(t0, color="0.5", ls=":", label="first damage")
ax.set(title=f"episode {ep_id}: raw vs. trimmed and downsampled", xlabel="time (s)", ylabel="health")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()

### 6.3 The derivative is the hard part

Health drops in chunks, one shot at a time, so a plain finite difference is mostly zeros and
spikes. SINDy fits whatever derivative it is given, so it is smoothed first (Savitzky–Golay,
`SMOOTH_WINDOW` points). Notebook 04 Parts 4–5 check that the fitted equations barely depend on
this choice, and show what goes wrong when the data is noisy.

In [ ]:
traj = trajectories[ep_id]
x, t = traj[STATE_COLUMNS].to_numpy(), traj["t"].to_numpy()
raw = ps.FiniteDifference()(x, t=t)
smooth = ps.SmoothedFiniteDifference(smoother_kws={"window_length": SMOOTH_WINDOW, "polyorder": 2})(x, t=t)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t, raw[:, 0], color="0.7", lw=0.8, label="finite difference")
ax.plot(t, smooth[:, 0], color=TEAM_COLORS["blue"], label=f"smoothed (window {SMOOTH_WINDOW})")
ax.set(title=f"d(health_blue)/dt, episode {ep_id}", xlabel="time since engagement (s)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

### 6.4 Fit three candidate models

- SINDy, degree 1, threshold 0.1: the model notebook 04 currently picks,
- SINDy, degree 2, threshold 0.005: the richer model of the earlier 5v5 runs,
- the exact Lanchester square law.

`LanchesterModel` mimics the SINDy interface, so the same evaluation code runs on all three.

In [ ]:
models = {
    "SINDy deg 1, thr 0.1": fit_sindy(trajectories, train_ids, STATE_COLUMNS, dt_fit, SMOOTH_WINDOW, 1, 0.1),
    "SINDy deg 2, thr 0.005": fit_sindy(
        trajectories, train_ids, STATE_COLUMNS, dt_fit, SMOOTH_WINDOW, 2, 0.005
    ),
    "Lanchester": fit_lanchester(trajectories, train_ids, STATE_COLUMNS, dt_fit, SMOOTH_WINDOW),
}
for name, m in models.items():
    print(name)
    for s, eq in zip(STATE_COLUMNS, m.equations(precision=3), strict=True):
        print(f"  ({s})' = {eq}")

### 6.5 How a model is judged

`summarize_eval` integrates the model on every **test** episode, from two starting points:

- **from the start** of the engagement. Every episode starts from (nearly) the same state, so a
  deterministic model predicts the same, average battle each time. It cannot beat
  `majority_baseline` (always guess the more common winner) at picking winners.
- **from midway**, from the real state halfway through. Here episodes differ, so this tests
  whether the model understands how the current balance decides the rest. Baseline:
  `leader_baseline_midway`, "whoever has more health now wins".

Main metrics (simulate **until elimination**, i.e. until one side drops to 0.02):

| metric | meaning |
|---|---|
| `winner_acc_start_elim`, `winner_acc_midway_elim` | share of test episodes where the side the model eliminates first really lost |
| `duration_mae_start`, `duration_mae_midway` | mean error of the predicted remaining battle time (s) |
| `no_elimination` | simulations where nobody was eliminated within 3× the longest episode (counted as wrong) |
| `rmse_start`, `rmse_midway` | how closely the simulated trajectory follows the real one, over the real time span |
| `n_terms` | non-zero coefficients: model complexity |

The older `winner_acc_start`/`winner_acc_midway` stop at the real end time and pick the side with
more health. They use the real battle length, which the model would not know in practice.

In [ ]:
MAIN_METRICS = [
    "winner_acc_start_elim",
    "majority_baseline",
    "winner_acc_midway_elim",
    "leader_baseline_midway",
    "duration_mae_start",
    "duration_mae_midway",
    "no_elimination",
    "rmse_midway",
    "winner_acc_midway",
    "n_terms",
]
results = pd.DataFrame(
    {
        name: summarize_eval(m, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])
        for name, m in models.items()
    }
).T
results[MAIN_METRICS].astype(float).round(3)

**How to read it:** a model is only useful from midway if `winner_acc_midway_elim` beats
`leader_baseline_midway`; matching it means the model has learned "the leader wins", nothing more.
`duration_mae_*` says whether the model gets the *pace* of the battle right, which the old metric
never tested.

In [ ]:
test_won = [i for i in test_ids if index.loc[i, "battle_won"]][:2]
test_lost = [i for i in test_ids if not index.loc[i, "battle_won"]][:2]
for name, m in models.items():
    plot_predictions(m, test_won + test_lost, trajectories, STATE_COLUMNS, index["battle_won"], title=name)
    plt.show()

### 6.6 Simulating until elimination, step by step

What `simulate_to_elimination` does for one test episode, from midway: integrate forward with no
knowledge of the real end time, stop when a side reaches the threshold (dotted line), and compare
the predicted end (dashed) with the real one (solid grey).

In [ ]:
ep_id = test_lost[0]
traj = trajectories[ep_id]
mid = len(traj) // 2
x0 = traj[STATE_COLUMNS].iloc[mid].to_numpy()
remaining = traj["t"].iloc[-1] - traj["t"].iloc[mid]

fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 3.5), sharey=True)
for ax, (name, m) in zip(axes, models.items(), strict=True):
    blue_won, t_elim = simulate_to_elimination(m, x0, t_max=60.0)
    t_plot = np.arange(0, 1.3 * max(remaining, t_elim or remaining), dt_fit)
    sim = m.simulate(x0, t_plot, integrator_kws={"method": "LSODA"})
    t_after = traj["t"].iloc[mid:] - traj["t"].iloc[mid]
    for k, col in enumerate(STATE_COLUMNS):
        color = TEAM_COLORS[col.split("_")[-1]]
        ax.plot(t_after, traj[col].iloc[mid:], color=color, label=f"{col} actual")
        ax.plot(t_plot[: len(sim)], sim[:, k], "--", color=color, alpha=0.8)
    ax.axhline(0.02, color="k", ls=":", lw=0.8)
    ax.axvline(remaining, color="0.5", label="real end")
    if t_elim is not None:
        ax.axvline(t_elim, color="0.5", ls="--", label="predicted end")
    verdict = "no elimination" if blue_won is None else ("Blue wins" if blue_won else "Red wins")
    ax.set(title=f"{name}\npredicted: {verdict}", xlabel="seconds after midway", ylim=(-0.1, 1.05))
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("health")
axes[-1].legend(fontsize=7)
fig.suptitle(f"episode {ep_id} (Blue lost), real remaining time {remaining:.1f} s")
fig.tight_layout()
plt.show()

Look at what happens *after* the dashed line for the degree-1 SINDy model: its `+health_red`
self-term makes Red's health grow again once Blue is gone. Stopping at the first elimination
ignores that, but it is a sign the model is fitting the data window rather than physics.

### 6.7 Other states

`STATE_COLUMNS` can be any Blue/Red pair of Level A or engagement columns, e.g.
`["alive_blue", "alive_red"]` or `["engaged_blue", "engaged_red"]` (normalize engaged by the
number of units first). Notebook 06 compares these as Lanchester drivers; Part 9 summarizes it.

## Part 7 · Saved runs (`runs/`)

`save_run` stores a fitted model once it is worth keeping: `config.json` (dataset, split,
preprocessing, model choices, git commit), `model.json`, `equations.txt`, `metrics.json`,
`per_episode.csv` and figures. All runs so far, newest last (runs saved before the elimination
metric existed have no `*_elim` values):

In [ ]:
run_rows = []
for run_dir in sorted(p for p in RUNS_DIR.iterdir() if (p / "config.json").exists()):
    cfg = json.loads((run_dir / "config.json").read_text())
    met = json.loads((run_dir / "metrics.json").read_text())
    run_rows.append(
        {
            "run": run_dir.name,
            "dataset": cfg["dataset"]["name"],
            "state": "+".join(
                c.replace("_blue", "").replace("_red", "") for c in cfg["preprocessing"]["state_columns"][::2]
            ),
            "degree": cfg["model"].get("degree"),
            "threshold": cfg["model"].get("threshold"),
            **{
                k: met.get(k)
                for k in [
                    "n_terms",
                    "winner_acc_midway",
                    "leader_baseline_midway",
                    "winner_acc_midway_elim",
                    "duration_mae_midway",
                    "rmse_midway",
                ]
            },
            "equations": (run_dir / "equations.txt").read_text().strip().replace("\n", "  |  "),
        }
    )
runs_table = pd.DataFrame(run_rows).set_index("run")
runs_table.drop(columns="equations").round(3)

In [ ]:
for run, row in runs_table.iterrows():
    print(f"{run}  ({row.dataset})")
    for line in row.equations.split("  |  "):
        print("   ", line)

## Part 8 · Where everything lives

### Notebooks

| notebook | what it does |
|---|---|
| `00_recap.ipynb` | this overview |
| `01_models_explained.ipynb` | how the model code in `sindy_wm.models` works, function by function |
| `02_analysis/04_sweep_and_first_sindy.ipynb` | sweep overview of one experiment; Level A of one dataset; SINDy fit, model comparison, run saving; sensitivity to smoothing/downsampling; noise slider |
| `02_analysis/05_compare_exp01_exp02.ipynb` | fixed vs. randomized spawn: win rates, outcome variety, durations |
| `02_analysis/06_lanchester_drivers.ipynb` | exact Lanchester fit on health vs. alive vs. engaged |
| `02_analysis/read_experiment.ipynb` | scratch notebook |

### Every public function and class in `src/sindy_wm`

Generated from the code, so it is always current. The first line of each docstring; see the
source for arguments.

In [ ]:
import importlib
import pkgutil

api_rows = []
for info in pkgutil.walk_packages(sindy_wm.__path__, prefix="sindy_wm."):
    module = importlib.import_module(info.name)
    for name, obj in inspect.getmembers(module, lambda o: inspect.isfunction(o) or inspect.isclass(o)):
        if name.startswith("_") or obj.__module__ != module.__name__:
            continue
        doc = (inspect.getdoc(obj) or "").split("\n")[0]
        api_rows.append({"module": info.name.removeprefix("sindy_wm."), "name": name, "summary": doc})

with pd.option_context("display.max_rows", None, "display.max_colwidth", 110):
    display(pd.DataFrame(api_rows).set_index(["module", "name"]))

## Part 9 · Findings, open questions, loose ends

### What we know (details in the README's Findings)

**About the simulator**
- Blue's win rate rises with aggression and falls with `step_mul`: Red's AI reacts every tick,
  Blue only every `step_mul` ticks, and its random actions last a whole step.
- Battles last about the same number of seconds at every `step_mul` (Part 4).
- A fixed spawn plus aggression 1.0 replays one game; randomized spawns fix that (exp02, exp03).
- The fixed exp01 layout was unusually favourable to Blue; exp02's lower win rates are probably
  the more representative ones.

**About modelling**
- A model started from the beginning of the engagement can only predict the average battle; the
  test that matters is from midway, against "the leader wins".
- On health, the fitted equations are stable across smoothing windows and downsampling
  (notebook 04 Part 4), so the derivative estimate is not driving the result.
- Positions/distance are not worth adding to the state after trimming: the spawn distance mostly
  just shifts *when* the fight starts.
- Lanchester on `health` and `alive` lands exactly on the leader baseline; on `engaged` it gives a
  negative coefficient, because engaged units are not a depleting stock (notebook 06).
- On 5v5 (degree 2) SINDy beat the leader baseline from midway with the old metric
  (Part 7, e.g. 0.84 vs 0.73). On 10v10 (exp03, degree 1) it only ties it.
- With the elimination metric on `10m_attack_a090_s1_v1` (Part 6, as of writing): none of the
  three models beats the leader baseline from midway (0.93). Degree-1 SINDy ties it but predicts
  battles far too long from the start (Red never quite dies); degree 2 follows the trajectories
  much better (RMSE 0.03 vs 0.08, duration error under 1 s) but picks fewer winners (0.86).
  Following the average trajectory and calling close fights are different skills.

### Open questions
1. Does SINDy beat "the leader wins" **consistently**, with the elimination metric, across
   datasets and scenario sizes, or only on some?
2. Is the better 5v5 result about scenario size, or about the degree-2 library? (Fit degree 2 on
   exp03 and degree 1 on exp02 with the same settings.)
3. Which state is right: health is smooth but not what drives firepower; alive is mechanistic but
   a step function. A combined or engagement-aware state?
4. How to model the approach phase, which trimming currently throws away (a model per phase?).

### Loose ends in the repository
- `envs/scenarios.py` defines its own `PROJECT_ROOT`/`SCENARIO_DIR` instead of importing `paths`.
- `data/raw/exp01_5m_aggression_stepmul_test/` is empty.
- `runs/` has two pairs of duplicate runs (same dataset, settings and equations, saved twice).
- No held-out dataset with new seeds for final evaluation yet.

### A possible next step
Turn Part 6 into a loop over every exp02 and exp03 dataset with randomized spawns: fit the same
few models on each, and collect the elimination metrics into one table. That answers questions 1
and 2 with what is already collected, and it is the table that would go into the thesis.